# EDA 3 (Interim) - RECH23: Características de la Vivienda y Riqueza
Este notebook audita el módulo RECH23, el cual contiene las variables estructurales del hogar que actúan como determinantes subyacentes de la desnutrición crónica (riqueza, agua, saneamiento y materiales de vivienda).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mnp.config import INTERIM_DATA_DIR

# Configuración de estilo
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
DATA_PATH = INTERIM_DATA_DIR / "rech23_cleaned.parquet"
print(f"Cargando datos desde: {DATA_PATH}")
df_rech23 = pd.read_parquet(DATA_PATH)
print(f"Filas (Hogares totales): {len(df_rech23):,}")
print(f"Columnas: {len(df_rech23.columns)}")
df_rech23.head()

## 1. El Predictor Maestro: Índice de Riqueza (`HV270`)

In [ ]:
# Distribución del Índice de Riqueza Quintiles (Ordinal)
plt.figure(figsize=(10, 5))
# Ordenar explícitamente para respetar el ordinal de 1 (El más pobre) a 5 (El más rico)
sns.countplot(data=df_rech23, y='HV270', palette='RdYlGn', order=sorted(df_rech23['HV270'].dropna().unique()))
plt.title('Distribución de Riqueza en los Hogares Muestreados (HV270)')
plt.xlabel('Cantidad de Hogares')
plt.ylabel('Quintil de Riqueza (Ordinal)')
plt.show()

nulos_riq = round(df_rech23['HV270'].isna().mean() * 100, 2)
print(f"Porcentaje de nulos en Índice de Riqueza (HV270): {nulos_riq}%")

## 2. Infraestructura Sanitaria: Agua (`HV201`) y Baños (`HV205`)
En la literatura médica, el acceso a agua por tubería y baños conectados al desagüe previene la EDA (Enfermedad Diarreica Aguda), que es un detonante masivo de la desnutrición.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 12))

# 1. Fuente de Agua (HV201)
# Top 8 para que sea legible
top_agua = df_rech23['HV201'].value_counts().iloc[:8].index
sns.countplot(data=df_rech23[df_rech23['HV201'].isin(top_agua)], y='HV201', palette='Blues_r', ax=axes[0], order=top_agua)
axes[0].set_title('Top 8 - Fuentes de Agua para Beber (HV201)')
axes[0].set_xlabel('Hogares')

# 2. Servicio Higiénico (HV205)
top_banos = df_rech23['HV205'].value_counts().iloc[:8].index
sns.countplot(data=df_rech23[df_rech23['HV205'].isin(top_banos)], y='HV205', palette='Oranges_r', ax=axes[1], order=top_banos)
axes[1].set_title('Top 8 - Tipos de Servicio Higiénico (HV205)')
axes[1].set_xlabel('Hogares')

plt.tight_layout()
plt.show()

## 3. Materiales de la Vivienda: Piso (`HV213`) y Pared (`HV214`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Material del Piso (HV213)
top_pisos = df_rech23['HV213'].value_counts().iloc[:6].index
sns.countplot(data=df_rech23[df_rech23['HV213'].isin(top_pisos)], y='HV213', palette='rocket', ax=axes[0], order=top_pisos)
axes[0].set_title('Top 6 - Material del Piso (HV213)')

# Material de la Pared (HV214)
top_pared = df_rech23['HV214'].value_counts().iloc[:6].index
sns.countplot(data=df_rech23[df_rech23['HV214'].isin(top_pared)], y='HV214', palette='mako', ax=axes[1], order=top_pared)
axes[1].set_title('Top 6 - Material de las Paredes (HV214)')

plt.tight_layout()
plt.show()

## 4. Justificación de Variables Excluidas del Análisis Visual

El archivo `RECH23` contiene una inmensa cantidad de variables operativas adicionales (ej. material del techo, si la cocina está dentro de la casa, si tienen refrigeradora, radio, TV). Todas estas sub-variables ya están consolidadas matemáticamente por el INEI dentro del Índice de Riqueza (`HV270`), por lo que graficarlas individualmente sería redundante. Las conservamos por si el modelo de Machine Learning encuentra algún valor marginal, pero el predictor central es el `HV270`.